# TF-LUK LAB — Motor de investigación para Google Colab

Este notebook descarga datos EOD desde tu **Massive API Key**, hace un escaneo sectorial/EMA y un backtest simplificado, y crea **`report.json`** para tu web en Vercel.

**Importante:** señales experimentales propuestas a partir del estudio de Martin Luk, no operaciones suyas. Universo manual con sesgo de supervivencia. No sustituye backtest intradía.


## 1. Configuración del período y restricciones

In [ ]:
# TF-LUK LAB · CONFIGURACIÓN
# Cambia las fechas y la cantidad de acciones si quieres.
START_DATE = "2026-01-01" # Incluye 150 sesiones de calentamiento para EMA150.
END_DATE   = "2026-11-20" # Cambia por tu último cierre disponible.
MAX_SYMBOLS = 30           # Filas de universe.csv. Aumenta según cobertura y rate limit.
DELAY_SECONDS = 1.0        # Evita saturar la API. Reducir solo si tu plan lo permite.
MIN_DOLLAR_VOLUME = 10_000_000
HOLD_SESSIONS = 10
SLIPPAGE_BPS = 5          # 5 puntos básicos POR LADO (modelo simplificado).


## 2. Preparar Colab y conectar Massive de forma segura

In [ ]:
!pip -q install "pandas>=2.0,<3" "numpy>=1.24,<3" "requests>=2.31,<3"
from google.colab import files
from getpass import getpass
from pathlib import Path
import os, subprocess, sys
ROOT=Path('/content/TF-Luk-Lab')
(ROOT/'engine').mkdir(parents=True,exist_ok=True)
(ROOT/'data').mkdir(parents=True,exist_ok=True)
(ROOT/'cache').mkdir(parents=True,exist_ok=True)
API_KEY=getpass('Introduce tu clave de MASSIVE API (oculta): ').strip()
if not API_KEY: raise ValueError('Necesitas tu Massive API Key')
print('Clave recibida en memoria. Nunca se escribe en el JSON ni se imprime.')


## 3. Crear el motor y el universo de acciones

In [ ]:
# Esta celda incluye el motor COMPLETO, sin depender de un GitHub aún no publicado.
ENGINE_SOURCE = '"""TF-LUK LAB: educational EOD research engine, not Martin Luk\'s original code.\n\nProvider: Massive.com REST OHLC API. All signals are generated after close t,\nexecuted at NEXT session open t+1; when both target and stop touch a bar,\nstop is assumed to trigger first. Fixed present-day universe introduces\nsurvivorship bias; no intraday execution or spread orderbook modeled.\n"""\nfrom __future__ import annotations\nimport argparse, datetime as dt, io, json, os, pathlib, time\nfrom collections import Counter, defaultdict\nimport numpy as np\nimport pandas as pd\nimport requests\n\nROOT=pathlib.Path(__file__).resolve().parents[1]\nBASE=\'https://api.massive.com\'\nETF_SYMBOLS=[\'SPY\',\'QQQ\',\'IWM\',\'SMH\',\'IGV\',\'XBI\',\'XLE\',\'XLF\']\nSECTOR_ETF={\n    \'Semiconductores\':\'SMH\',\'Software y ciberseguridad\':\'IGV\',\n    \'IA e infraestructura\':\'QQQ\',\'Cuántica\':\'IWM\',\'Cripto\':\'QQQ\',\n    \'Fintech\':\'XLF\',\'Energía y solar\':\'XLE\',\'Biotecnología\':\'XBI\',\n    \'Consumo y tecnología\':\'QQQ\'\n}\nSETUPS={\n \'L1\':(\'LONG\',\'Pullback a EMA 9 ascendente\'),\n \'L2\':(\'LONG\',\'Undercut & reclaim\'),\n \'S1\':(\'SHORT\',\'Rechazo de EMA 9 descendente\'),\n \'S2\':(\'SHORT\',\'Failed breakout / upthrust\')\n}\n\ndef fetch_bars(session,ticker,start,end,api_key,cache):\n    cache.mkdir(parents=True,exist_ok=True)\n    path=cache/f\'{ticker}_{start}_{end}.json\'\n    if path.exists():\n        payload=json.loads(path.read_text(encoding=\'utf-8\'))\n    else:\n        url=f\'{BASE}/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}\'\n        params={\'adjusted\':\'true\',\'sort\':\'asc\',\'limit\':50000,\'apiKey\':api_key}\n        for attempt in range(6):\n            try:\n                r=session.get(url,params=params,timeout=35)\n                if r.status_code==429 or r.status_code>=500:\n                    time.sleep(min(60,4*2**attempt)); continue\n                if r.status_code in (401,403):\n                    raise RuntimeError(f\'{ticker}: acceso denegado ({r.status_code}). Revisa tu clave y cobertura Massive.\')\n                r.raise_for_status()\n                payload=r.json()\n                if payload.get(\'status\')==\'ERROR\':\n                    raise RuntimeError(f\'{ticker}: {payload.get("error",payload)}\')\n                path.write_text(json.dumps(payload),encoding=\'utf-8\')\n                break\n            except requests.RequestException:\n                if attempt==5: raise\n                time.sleep(min(30,2**attempt))\n        else: raise RuntimeError(f\'{ticker}: límite de intentos API\')\n    bars=payload.get(\'results\') or []\n    if not bars: return pd.DataFrame()\n    df=pd.DataFrame(bars)\n    for k in [\'o\',\'h\',\'l\',\'c\',\'v\',\'t\']:\n        if k not in df: return pd.DataFrame()\n    dates=pd.to_datetime(df[\'t\'],unit=\'ms\',utc=True).dt.tz_convert(\'America/New_York\').dt.tz_localize(None).dt.normalize()\n    out=pd.DataFrame({\'date\':dates,\'open\':pd.to_numeric(df.o,errors=\'coerce\'),\n        \'high\':pd.to_numeric(df.h,errors=\'coerce\'),\'low\':pd.to_numeric(df.l,errors=\'coerce\'),\n        \'close\':pd.to_numeric(df.c,errors=\'coerce\'),\'volume\':pd.to_numeric(df.v,errors=\'coerce\')})\n    out=out.dropna().drop_duplicates(\'date\').sort_values(\'date\').set_index(\'date\')\n    out=out[out[[\'open\',\'high\',\'low\',\'close\']].gt(0).all(axis=1)]\n    return out\n\ndef features(df):\n    x=df.copy()\n    for n in [9,21,50,150]: x[f\'ema{n}\']=x.close.ewm(span=n,adjust=False,min_periods=n).mean()\n    x[\'ret20\']=x.close.pct_change(20,fill_method=None)\n    x[\'ret5\']=x.close.pct_change(5,fill_method=None)\n    x[\'ret60\']=x.close.pct_change(60,fill_method=None)\n    x[\'dollarvol20\']=(x.close*x.volume).rolling(20,min_periods=20).mean()\n    x[\'prev_low\']=x.low.shift(1)\n    x[\'prev_high20\']=x.high.shift(1).rolling(20,min_periods=20).max()\n    wk=x.close.resample(\'W-FRI\').last().dropna()\n    wk9=wk.ewm(span=9,adjust=False,min_periods=9).mean()\n    # Only completed Fridays may appear at daily timestamps. No partial-week lookahead.\n    x[\'weekly_ema9\']=wk9.reindex(x.index,method=\'ffill\')\n    x[\'leading\']=(x.close>x.ema9)&(x.ema9>x.ema21)&(x.ema21>x.ema50)\n    x[\'lagging\']=(x.close<x.ema9)&(x.ema9<x.ema21)&(x.ema21<x.ema50)\n    return x\n\ndef load_universe(path,limit=None):\n    u=pd.read_csv(path).fillna(\'\')\n    required={\'ticker\',\'sector\'}\n    if not required.issubset(u.columns): raise ValueError(\'universe.csv requiere ticker,sector\')\n    u.ticker=u.ticker.str.strip().str.upper()\n    u.sector=u.sector.str.strip()\n    u=u[(u.ticker!=\'\')&(u.sector!=\'\')].drop_duplicates(\'ticker\')\n    if limit and limit>0: u=u.head(limit)\n    return dict(zip(u.ticker,u.sector))\n\ndef derive_signals(x, market, sector_history, sector, stock, min_dollarvol=1e7):\n    # All observations are fully known at signal-day CLOSE; entry is next-day OPEN.\n    s=x.copy()\n    spy=market[\'SPY\'].reindex(s.index)\n    qqq=market[\'QQQ\'].reindex(s.index)\n    iwm=market[\'IWM\'].reindex(s.index)\n    sector_df=sector_history[sector].reindex(s.index)\n    s[\'rs20\']=s.ret20-spy.ret20\n    s[\'sector_rs20\']=sector_df[\'ret20\']-spy.ret20\n    s[\'market_long\']=(spy.close>spy.ema21)&(qqq.close>qqq.ema21)&(iwm.close>iwm.ema50)\n    s[\'market_short\']=(spy.close<spy.ema21)|(qqq.close<qqq.ema21)|(iwm.close<iwm.ema9)\n    liquid=s.dollarvol20>=min_dollarvol\n    sector_long=s.sector_rs20>0\n    sector_short=s.sector_rs20<0\n    ma_up=(s.ema9>s.ema21)&(s.ema21>s.ema50)&(s.ema9>s.ema9.shift(5))\n    ma_down=(s.ema9<s.ema21)&(s.ema21<s.ema50)&(s.ema9<s.ema9.shift(5))\n    s[\'L1\']=liquid&s.market_long&sector_long&(s.rs20>0)&ma_up&(s.low<=s.ema9*1.012)&(s.close>=s.ema9)&(s.close>s.open)\n    s[\'L2\']=liquid&s.market_long&sector_long&(s.rs20>0)&(s.close>s.ema21)&(s.low<s.prev_low)&(s.close>s.prev_low)&(s.close>s.open)\n    s[\'S1\']=liquid&s.market_short&sector_short&(s.rs20<0)&ma_down&(s.high>=s.ema9*.988)&(s.close<s.ema9)&(s.close<s.open)\n    s[\'S2\']=liquid&s.market_short&sector_short&(s.rs20<0)&(s.high>s.prev_high20)&(s.close<s.prev_high20)&(s.close<s.open)&(s.close<s.ema9)\n    return s\n\ndef resolve_trade(x,signal_index,setup,max_hold=10,rr_target=3.0,slippage_bps=5.0):\n    direction,description=SETUPS[setup]\n    if signal_index+1>=len(x): return None\n    row=x.iloc[signal_index]\n    entry_date=x.index[signal_index+1]\n    first=x.iloc[signal_index+1]\n    slip=slippage_bps/10000.0\n    entry=float(first.open)*(1+slip if direction==\'LONG\' else 1-slip)\n    stop=float(row.low)*.995 if direction==\'LONG\' else float(row.high)*1.005\n    risk=entry-stop if direction==\'LONG\' else stop-entry\n    if not np.isfinite(risk) or risk<=0 or risk/entry>.10 or risk/entry<.002: return None\n    target=entry+(rr_target*risk if direction==\'LONG\' else -rr_target*risk)\n    final_i=min(len(x)-1,signal_index+max_hold)\n    mfe=0.0; mae=0.0\n    exit_price=None; outcome=\'TIEMPO\'; exit_date=None\n    for j in range(signal_index+1,final_i+1):\n        bar=x.iloc[j]\n        if direction==\'LONG\':\n            mfe=max(mfe,(float(bar.high)-entry)/risk)\n            mae=max(mae,(entry-float(bar.low))/risk)\n            # Stops take precedence when both touched. Gaps fill at OPEN, not the stop.\n            if bar.open<=stop: exit_price=float(bar.open); outcome=\'GAP_STOP\'\n            elif bar.low<=stop: exit_price=stop; outcome=\'STOP\'\n            elif bar.open>=target: exit_price=float(bar.open); outcome=\'GAP_TARGET\'\n            elif bar.high>=target: exit_price=target; outcome=\'TARGET\'\n        else:\n            mfe=max(mfe,(entry-float(bar.low))/risk)\n            mae=max(mae,(float(bar.high)-entry)/risk)\n            if bar.open>=stop: exit_price=float(bar.open); outcome=\'GAP_STOP\'\n            elif bar.high>=stop: exit_price=stop; outcome=\'STOP\'\n            elif bar.open<=target: exit_price=float(bar.open); outcome=\'GAP_TARGET\'\n            elif bar.low<=target: exit_price=target; outcome=\'TARGET\'\n        if exit_price is not None:\n            exit_date=x.index[j]; break\n    if exit_price is None:\n        exit_price=float(x.iloc[final_i].close); exit_date=x.index[final_i]\n    exit_price*=1-slip if direction==\'LONG\' else 1+slip\n    r=(exit_price-entry)/risk if direction==\'LONG\' else (entry-exit_price)/risk\n    return dict(ticker=x.attrs.get(\'ticker\',\'\'),setup=setup,direction=direction,\n        signal_date=x.index[signal_index].strftime(\'%Y-%m-%d\'),\n        entry_date=entry_date.strftime(\'%Y-%m-%d\'),exit_date=exit_date.strftime(\'%Y-%m-%d\'),\n        entry=round(entry,4),stop=round(stop,4),target=round(target,4),\n        exit=round(exit_price,4),r=round(float(r),4),mfe_r=round(mfe,4),mae_r=round(mae,4),reason=outcome)\n\ndef summarize(trades):\n    if not trades: return {\'n\':0,\'win_rate\':None,\'avg_r\':None,\'profit_factor\':None,\'total_r\':None,\'max_drawdown_r\':None}\n    a=np.array([t[\'r\'] for t in trades],float)\n    gains=float(a[a>0].sum()); losses=float(-a[a<0].sum())\n    eq=np.cumsum(a)\n    dd=eq-np.maximum.accumulate(np.r_[0.0,eq])[1:]\n    return {\'n\':len(a),\'win_rate\':round(float((a>0).mean()*100),2),\n      \'avg_r\':round(float(a.mean()),3),\n      \'profit_factor\':round(gains/losses,2) if losses>0 else None,\n      \'total_r\':round(float(a.sum()),2),\'max_drawdown_r\':round(float(dd.min()),2),\n      \'avg_mfe_r\':round(float(np.mean([t[\'mfe_r\'] for t in trades])),2),\n      \'avg_mae_r\':round(float(np.mean([t[\'mae_r\'] for t in trades])),2)}\n\ndef run_engine(api_key,universe_file=ROOT/\'universe.csv\',out=ROOT/\'data/report.json\',\n               start=\'2026-01-01\',end=None,max_symbols=30,delay=.25,\n               min_dollarvol=10_000_000,hold=10,slippage_bps=5.0, cache=ROOT/\'cache\'):\n    end=end or dt.datetime.now(dt.timezone.utc).date().isoformat()\n    u=load_universe(universe_file,max_symbols)\n    wanted=list(dict.fromkeys(list(u)+ETF_SYMBOLS))\n    session=requests.Session()\n    prices={}; failed={}\n    print(f\'Buscando históricos de {len(wanted)} tickers: {start} → {end}\')\n    for i,t in enumerate(wanted,1):\n        try:\n            x=fetch_bars(session,t,start,end,api_key,pathlib.Path(cache))\n            if len(x)<80: failed[t]=f\'solo {len(x)} velas\'; continue\n            prices[t]=features(x)\n            print(f\'  [{i}/{len(wanted)}] {t}: {len(x)} velas\')\n        except Exception as ex:\n            if \'acceso denegado\' in str(ex): raise\n            failed[t]=str(ex)\n            print(f\'  [{i}/{len(wanted)}] {t}: ERROR {str(ex)[:100]}\')\n        if delay: time.sleep(delay)\n    if not all(t in prices for t in [\'SPY\',\'QQQ\',\'IWM\']):\n        raise RuntimeError(\'Faltan datos de SPY/QQQ/IWM: no se puede calcular el régimen de mercado. \'+str(failed))\n    actual={t:s for t,s in u.items() if t in prices}\n    if not actual: raise RuntimeError(\'No hay acciones con datos suficientes. Revisar clave, plan y fechas.\')\n    sector_symbols=defaultdict(list)\n    for t,sec in actual.items(): sector_symbols[sec].append(t)\n    # Historical equal-weight sector return (fixed-current-universe approximation).\n    sec_history={}\n    for sec,ts in sector_symbols.items():\n        returns=pd.concat({t:prices[t][\'ret20\'] for t in ts},axis=1)\n        sec_history[sec]=pd.DataFrame({\'ret20\':returns.mean(axis=1,skipna=True)})\n    indices={t:prices[t] for t in [\'SPY\',\'QQQ\',\'IWM\']}\n    all_dates=sorted(set(prices[\'SPY\'].index))\n    latest_date=max(all_dates)\n    stock_features={}\n    for t,sec in actual.items():\n        x=derive_signals(prices[t],indices,sec_history,sec,t,min_dollarvol=min_dollarvol)\n        x.attrs[\'ticker\']=t\n        stock_features[t]=x\n    # Select all signal candidates for testing; no possible intraday same-bar hindsight.\n    trades=[]; signals=[]\n    for t,x in stock_features.items():\n        sec=actual[t]\n        for i in range(80,len(x)):\n            d=x.index[i]\n            if d>latest_date: continue\n            for setup in SETUPS:\n                if not bool(x.iloc[i][setup]): continue\n                if d==latest_date:\n                    r=x.iloc[i]; stop=float(r.low)*.995 if setup.startswith(\'L\') else float(r.high)*1.005\n                    signals.append(dict(ticker=t,sector=sec,setup=setup,direction=SETUPS[setup][0],\n                        description=SETUPS[setup][1],signal_date=d.strftime(\'%Y-%m-%d\'),\n                        last_close=round(float(r.close),4),stop_reference=round(stop,4),\n                        ret20=round(float(r.ret20)*100,2),sector_rs20=round(float(r.sector_rs20)*100,2),\n                        dollarvol20=round(float(r.dollarvol20),0),\n                        distance_to_stop_pct=round(abs(r.close-stop)/r.close*100,2),\n                        trend=\'Líder\' if r.leading else \'Rezagada\' if r.lagging else \'Intermedia\',\n                        chart=[round(float(c),3) for c in x.close.tail(36)]))\n                trade=resolve_trade(x,i,setup,max_hold=hold,slippage_bps=slippage_bps)\n                if trade and trade[\'exit_date\']<=latest_date.strftime(\'%Y-%m-%d\'):\n                    trade[\'sector\']=sec\n                    trades.append(trade)\n    trades=sorted(trades,key=lambda x:(x[\'entry_date\'],x[\'ticker\'],x[\'setup\']))\n    # Dedup same ticker same entry date when 2 setups coincide; keep both only as research if selected flag.\n    trades_unique=[]; seen=set()\n    for t in trades:\n        key=(t[\'ticker\'],t[\'entry_date\'],t[\'direction\'])\n        if key not in seen: trades_unique.append(t);seen.add(key)\n    trades=trades_unique\n    asof=latest_date.strftime(\'%Y-%m-%d\')\n    sectors=[]\n    for sec,ts in sector_symbols.items():\n        recs=[]\n        for t in ts:\n            x=stock_features[t]; x=x.loc[:latest_date]\n            if x.empty: continue\n            r=x.iloc[-1]\n            if (latest_date-x.index[-1]).days>5: continue\n            if np.isfinite(r.ret20): recs.append((t,r))\n        if not recs: continue\n        avg20=float(np.mean([float(r.ret20) for _,r in recs]))\n        avg5=float(np.mean([float(r.ret5) for _,r in recs if np.isfinite(r.ret5)]))\n        avg60=float(np.mean([float(r.ret60) for _,r in recs if np.isfinite(r.ret60)])) if any(np.isfinite(r.ret60) for _,r in recs) else 0\n        leaders=sum(bool(r.leading) for _,r in recs); laggards=sum(bool(r.lagging) for _,r in recs)\n        spyret=prices[\'SPY\'].loc[:latest_date].iloc[-1].ret20\n        sectors.append(dict(sector=sec,etf_proxy=SECTOR_ETF.get(sec,\'SPY\'),\n          ret5=round(avg5*100,2),ret20=round(avg20*100,2),ret60=round(avg60*100,2),\n          rs20=round((avg20-float(spyret))*100,2),leading=leaders,lagging=laggards,\n          stocks=len(recs),tickers=\', \'.join(t for t,_ in recs)))\n    sectors.sort(key=lambda s:s[\'rs20\'],reverse=True)\n    breadth={\'leading\':sum(bool(x.loc[:latest_date].iloc[-1].leading) for x in stock_features.values()),\n      \'lagging\':sum(bool(x.loc[:latest_date].iloc[-1].lagging) for x in stock_features.values()),\n      \'total\':len(stock_features)}\n    breadth[\'intermediate\']=breadth[\'total\']-breadth[\'leading\']-breadth[\'lagging\']\n    benchmarks=[]\n    for t in [\'SPY\',\'QQQ\',\'IWM\']:\n        r=prices[t].loc[:latest_date].iloc[-1]\n        benchmarks.append(dict(ticker=t,close=round(float(r.close),2),ret20=round(float(r.ret20)*100,2),\n           over_ema9=bool(r.close>r.ema9),over_ema21=bool(r.close>r.ema21),over_ema50=bool(r.close>r.ema50),\n           chart=[round(float(c),2) for c in prices[t].close.tail(42)]))\n    by_setup={k:summarize([t for t in trades if t[\'setup\']==k]) for k in SETUPS}\n    by_month={m:summarize([t for t in trades if t[\'entry_date\'].startswith(m)])\n              for m in sorted(set(t[\'entry_date\'][:7] for t in trades))}\n    equity=[];cum=0\n    for t in trades:\n        cum+=t[\'r\'];equity.append({\'date\':t[\'entry_date\'],\'cum_r\':round(cum,2)})\n    latest_stocks=[]\n    for t,x in stock_features.items():\n        x=x.loc[:latest_date];r=x.iloc[-1]\n        if (latest_date-x.index[-1]).days>5: continue\n        latest_stocks.append(dict(ticker=t,sector=actual[t],close=round(float(r.close),2),\n           ret5=round(float(r.ret5)*100,2) if pd.notna(r.ret5) else None,\n           ret20=round(float(r.ret20)*100,2) if pd.notna(r.ret20) else None,\n           rs20=round(float(r.rs20)*100,2) if pd.notna(r.rs20) else None,\n           trend=\'Líder\' if r.leading else \'Rezagada\' if r.lagging else \'Intermedia\',\n           dollarvol20=round(float(r.dollarvol20),0) if pd.notna(r.dollarvol20) else 0,\n           ema9=round(float(r.ema9),2) if pd.notna(r.ema9) else None,\n           ema21=round(float(r.ema21),2) if pd.notna(r.ema21) else None,\n           ema50=round(float(r.ema50),2) if pd.notna(r.ema50) else None,\n           chart=[round(float(c),2) for c in x.close.tail(42)]))\n    latest_stocks.sort(key=lambda r:r[\'rs20\'] if r[\'rs20\'] is not None else -999,reverse=True)\n    signals.sort(key=lambda r:(r[\'direction\']!=\'LONG\',-abs(r[\'sector_rs20\'])))\n    report={\n        \'meta\':{\'app\':\'TF-LUK LAB\',\'version\':\'1.0\',\'mode\':\'real\',\'provider\':\'Massive API\',\n         \'asof\':asof,\'start\':start,\'requested_end\':end,\'universe_count\':len(actual),\n         \'generated_utc\':dt.datetime.now(dt.timezone.utc).isoformat(),\n         \'data_note\':\'Datos ajustados de Massive; no incluyen tape, bid/ask ni avisos de earnings.\',\n         \'bias_note\':\'Universo definido hoy, sin tickers deslistados: sesgo de supervivencia. Sector por lista manual. Backtest señales al cierre/entrada siguiente apertura. Sin simultaneidad de cartera.\',\n         \'cost_note\':f\'{slippage_bps} bps de deslizamiento por lado; comisiones, borrow fees, borrow availability, impuestos y financiación no incluidos.\'},\n        \'market\':{\'benchmarks\':benchmarks,\'breadth\':breadth},\n        \'sectors\':sectors,\'stocks\':latest_stocks,\'candidates\':signals,\n        \'backtest\':{\'overall\':summarize(trades),\'by_setup\':by_setup,\'by_month\':by_month,\'trades\':trades[-800:],\n          \'equity_r\':equity[-800:],\'parameters\':{\'max_hold_sessions\':hold,\'target_r\':3.0,\n               \'slippage_bps_each_side\':slippage_bps,\'min_dollarvol20\':min_dollarvol,\n               \'entry\':\'siguiente apertura\',\'intrabar_collision\':\'stop primero\',\n               \'note\':\'R acumulado NO es retorno de cartera. Operaciones independientes pueden solaparse.\'}},\n        \'failed\':failed,\n        \'methodology\':{\'sources\':\'Transcripciones de Martin Luk, junio–noviembre 2026. Reglas exactas no publicadas.\',\n            \'long\':[\'L1: pullback a EMA 9 ascendiendo en acción y sector fuerte\',\'L2: undercut & reclaim sobre soporte\'],\n            \'short\':[\'S1: rechazo de EMA 9 descendente en sector débil\',\'S2: failed breakout + vela bajista\'],\n            \'disclaimer\':\'Hipótesis investigables inspiradas en observaciones. No señales del trader.\'}}\n    out=pathlib.Path(out);out.parent.mkdir(parents=True,exist_ok=True)\n    out.write_text(json.dumps(report,ensure_ascii=False,allow_nan=False,indent=2),encoding=\'utf-8\')\n    print(f\'✅ {out} | {len(actual)} símbolos | {len(signals)} setups activos | {len(trades)} operaciones históricas\')\n    return report\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--start\',default=\'2026-01-01\')\n    p.add_argument(\'--end\',default=None)\n    p.add_argument(\'--max-symbols\',type=int,default=30)\n    p.add_argument(\'--delay\',type=float,default=.25)\n    p.add_argument(\'--min-dollarvol\',type=float,default=10000000)\n    p.add_argument(\'--hold\',type=int,default=10)\n    p.add_argument(\'--slippage-bps\',type=float,default=5)\n    p.add_argument(\'--output\',default=str(ROOT/\'data/report.json\'))\n    p.add_argument(\'--universe\',default=str(ROOT/\'universe.csv\'))\n    args=p.parse_args()\n    key=os.getenv(\'MASSIVE_API_KEY\',\'\').strip()\n    if not key: raise SystemExit(\'Falta MASSIVE_API_KEY. En Colab se pide por getpass; en GitHub Actions crea un Secret.\')\n    run_engine(key,args.universe,args.output,args.start,args.end,args.max_symbols,args.delay,args.min_dollarvol,args.hold,args.slippage_bps)\n\nif __name__==\'__main__\': main()\n'
UNIVERSE_CSV = 'ticker,sector\nNVDA,Semiconductores\nAMD,Semiconductores\nMU,Semiconductores\nSNDK,Semiconductores\nINTC,Semiconductores\nAVGO,Semiconductores\nMRVL,Semiconductores\nTSM,Semiconductores\nGLW,IA e infraestructura\nAPLD,IA e infraestructura\nVRT,IA e infraestructura\nNBIS,IA e infraestructura\nCRWV,IA e infraestructura\nIREN,IA e infraestructura\nORCL,Software y ciberseguridad\nCRWD,Software y ciberseguridad\nNET,Software y ciberseguridad\nOKTA,Software y ciberseguridad\nCRM,Software y ciberseguridad\nRDDT,Software y ciberseguridad\nPATH,Software y ciberseguridad\nRBLX,Software y ciberseguridad\nRGTI,Cuántica\nQBTS,Cuántica\nIONQ,Cuántica\nHOOD,Fintech\nSOFI,Fintech\nXYZ,Fintech\nCOIN,Cripto\nMSTR,Cripto\nMARA,Cripto\nRIOT,Cripto\nFSLR,Energía y solar\nSEDG,Energía y solar\nSMR,Energía y solar\nOKLO,Energía y solar\nTSLA,Consumo y tecnología\nHIMS,Consumo y tecnología\nPLTR,IA e infraestructura\nBE,Energía y solar\nCVNA,Consumo y tecnología\nCELH,Consumo y tecnología\nXBI,Biotecnología\n'
(ROOT/'engine/luk_engine.py').write_text(ENGINE_SOURCE,encoding='utf-8')
(ROOT/'universe.csv').write_text(UNIVERSE_CSV,encoding='utf-8')
print('Motor listo. Para agregar acciones, cambia UNIVERSE_CSV o sube tu propio CSV.')


## 4. Ejecutar el escáner y backtest EOD

In [ ]:
# Ejecuta una descarga y backtest reproducible de datos diarios.
# Es normal que según el plan Massive haya límites de solicitudes o de profundidad histórica.
env={**os.environ,'MASSIVE_API_KEY':API_KEY}
cmd=[sys.executable,str(ROOT/'engine/luk_engine.py'),
     '--start',START_DATE,'--end',END_DATE,'--max-symbols',str(MAX_SYMBOLS),
     '--delay',str(DELAY_SECONDS),'--hold',str(HOLD_SESSIONS),
     '--slippage-bps',str(SLIPPAGE_BPS),'--min-dollarvol',str(MIN_DOLLAR_VOLUME)]
res=subprocess.run(cmd,cwd=str(ROOT),env=env)
if res.returncode!=0:
    raise RuntimeError('El motor no pudo terminar. Revisa clave, plan, fechas y errores impresos arriba.')
from IPython.display import display,Markdown
import json
report=json.loads((ROOT/'data/report.json').read_text(encoding='utf-8'))
summary=report['backtest']['overall']
print('Fecha de corte:',report['meta']['asof'])
print('Acciones válidas:',report['meta']['universe_count'])
print('Setups al cierre:',len(report['candidates']))
print('Operaciones históricas:',summary['n'])
print('Media R:',summary['avg_r'],'| Win rate:',summary['win_rate'],'% | Profit factor:',summary['profit_factor'])
print('Limitaciones:',report['meta']['bias_note'])


## 5. Descargar y publicar los resultados

Sube el **`report.json`** descargado en la ruta **`data/report.json`** de tu repositorio GitHub. En el próximo despliegue Vercel mostrará resultados reales. **Nunca publiques tu clave Massive.**

In [ ]:
# Descarga report.json y súbelo al repositorio GitHub en data/report.json.
# Cada vez que lo actualices, Vercel desplegará la nueva versión automáticamente.
files.download(str(ROOT/'data/report.json'))


## Mejoras futuras

- Agregar histórico **point-in-time** con deslistadas.
- Descargar 1/5/15 minutos con VWAP anclado.
- Validar spreads, fees, simultaneidad de cartera y exposición por sector.
- Comparar walk-forward y datos fuera de muestra.

Documentación Massive: https://massive.com/docs/rest/stocks/aggregates/custom-bars